# Agriculture Project Overview

## Introduction
This project focuses on analyzing agricultural data to gain insights into various factors affecting farming households' engagement in agricultural activities. By leveraging machine learning algorithms, the goal is to develop models that can predict the number of households engaged primarily in farm activities based on various socio-economic and agricultural indicators.

## Dataset
The dataset used in this project contains village-wise survey data collected as part of the Mission Antyodaya initiative in 2020. It includes information on socio-economic factors, agricultural practices, and infrastructure availability across different villages.

## Methodology

### Data Preprocessing
- The dataset underwent thorough preprocessing steps, including handling missing values, encoding categorical variables, and standardizing numerical features.
- Categorical variables were transformed into dummy variables to facilitate model training.

In [ ]:
# Imports for data manipulation and analysis
import re
import numpy as np
import pandas as pd

# Imports for visualization
import seaborn as sns
import matplotlib.pyplot as plt

# Imports for statistical analysis
import statsmodels.api as sm
import scipy.stats as stats

# Imports for machine learning
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, ElasticNet, Ridge, Lasso
from sklearn.metrics import mean_squared_error

# Imports for statistical tests and analysis
import statsmodels.stats.multicomp as mc
from statsmodels.formula.api import ols
from statsmodels.multivariate.manova import MANOVA

# Imports for progress visualization and model serialization
from tqdm import tqdm
from joblib import dump, load

# Function to get variable name
def get_var_name(var):
    for name, value in globals().items():
        if value is var:
            return name

In [ ]:
# Read the CSV file into a pandas DataFrame
df_main = pd.read_csv(r'village_wise_survey_data_mission_antyodaya_2020.csv')

In [ ]:
# Drop specified columns from the DataFrame
df_main.drop(columns=["SUB DISTRICT CODE", "BLOCK CODE", "GP CODE", "VILLAGE CODE", "VILLAGE PIN CODE", "STATE NAME", 
                      "DISTRICT NAME", "SUB DISTRICT NAME", "BLOCK NAME", "GP NAME", "VILLAGE NAME", "PC CODE", 
                      "AC CODE", "OTHER ASSEMBLY CONSTITUENCIES", "NUMBER OF HOUSEHOLDS ENGAGED MAJORLY IN NON-FARM ACTIVITIES"],
             inplace=True)
# Display the first few rows of the DataFrame
df_main.head()


In [ ]:
# Get the index of the specified column 
endcol = df_main.columns.get_loc("DOES THE VILLAGE HAVE LIVESTOCK EXTENSION SERVICES")
print(endcol)

In [ ]:
# Select columns up to the specified index
df = df_main.iloc[:, :endcol]

# Display the first few rows of the DataFrame
df.head()

In [ ]:
# Map values in the column "DOES THE VILLAGE HAS ANY FARMERS COLLECTIVE" to simplified categories
df["DOES THE VILLAGE HAS ANY FARMERS COLLECTIVE"] = df["DOES THE VILLAGE HAS ANY FARMERS COLLECTIVE"].map({
    "Primary Agriculture Cooperative Society(PACS)": "PACS",
    "Farmers Produce Organization(FPOs)": "FPO",
    "Both": "Both"
})

# Map values in the column "MAIN SOURCE OF IRRIGATION" to simplified categories
df["MAIN SOURCE OF IRRIGATION"] = df["MAIN SOURCE OF IRRIGATION"].map({
    "Ground water (tube well/well/pump)": "Ground Water",
    "Other": "Other",
    "Canals": "Canals",
    "Surface Water": "Surface Water"
})

In [ ]:
# Create new columns 'Area', 'Kharif', 'Rabi', and 'Others' initialized with NaN values
df["Area"] = np.nan
df["Kharif"] = np.nan
df["Rabi"] = np.nan
df["Others"] = np.nan

# Extract numbers from the string in the specified column
numbers = df["NET SOWN AREA (IN HECTARES) , IF IN ACRES DIVIDE BY 2.47"].str.findall(r'\d+\.*\d*').apply(lambda x: [float(i) for i in x])

# Assign extracted numbers to respective columns in the DataFrame
df["Area"] = numbers.str[0]
df["Kharif"] = numbers.str[1]
df["Rabi"] = numbers.str[2]
df["Others"] = numbers.str[3]

In [ ]:
# Define a list of columns and create a dictionary for mapping values
cols = ['availability of warehouse for food grain storage ', 'availability of soil testing centres', 'availability of fertilizer shop']
dic = dict(zip(df['availability of fertilizer shop'.upper()].value_counts().index.to_list(), ['>10', '5-10', '2-5', 'Yes', '1-2', '<1']))

# Map values in specified columns to simplified categories using the dictionary
for col in cols:
    col = col.upper()  # Convert column name to uppercase
    df[col] = df[col].map(dic)

# Drop the column "NET SOWN AREA (IN HECTARES) , IF IN ACRES DIVIDE BY 2.47" from the DataFrame
df = df.drop(columns="NET SOWN AREA (IN HECTARES) , IF IN ACRES DIVIDE BY 2.47")

In [ ]:
# Get the list of current column names and generate new column names
cols = df.columns.to_list()
new_cols = [re.sub(r'[^a-zA-Z0-9_]', '_', a.strip().lower()) for a in cols]

# Rename columns in the DataFrame using the new column names
df.rename(columns=dict(zip(cols, new_cols)), inplace=True)

# Rename specific columns in the DataFrame
df.rename(columns={'total_area_irrigated__in_hectare___if_in_acres_divide_by_2_47': 'total_area_irrigated__in_hectare',
                    'total_unirrigated_land_area__in_hectares___if_in_acres_divide_by_2_47': 'total_unirrigated_land_area__in_hectares'},
                    inplace=True)


In [ ]:
# Check for columns with missing values
for col in df.columns:
    if df[col].isna().sum() != 0:
        print(col)

In [ ]:
# Fill missing values with 'None'
df.fillna('None', inplace=True)

In [ ]:
# Check for columns with missing values
for col in df.columns:
    if df[col].isna().sum() != 0:
        print(col)

In [ ]:
# Save the DataFrame to a CSV file without including the index
df.to_csv("Agriculture.csv", index=False)

# Read the saved CSV file into a new DataFrame
df = pd.read_csv("Agriculture.csv")

# Display the first few rows of the DataFrame
df.head()

In [ ]:
# Join column names into a single string separated by "+"
string = "+".join(df.drop('number_of_households_engaged_majorly_in_farm_activities', axis=1).columns.to_list())

# Perform Ordinary Least Squares (OLS) regression
eq = ols(f'number_of_households_engaged_majorly_in_farm_activities~{string}', data=df).fit()

# Perform ANOVA and extract p-values
p_value = sm.stats.anova_lm(eq, typ=1)

In [ ]:
# Filter significant p-values (less than 0.05) and sort by ascending order
p_value_sig = p_value[p_value['PR(>F)'] < 0.05].sort_values(by='PR(>F)', ascending=True)

# Display the filtered p-values
display(p_value_sig)

# Get the list of significant columns
cols_sig = p_value_sig.index.tolist()

In [ ]:
# Add the target column to the list of significant columns again
cols_sig.append("number_of_households_engaged_majorly_in_farm_activities")

# Create a new DataFrame containing only the significant columns
df_sig = df[cols_sig]

# Display the first few rows of the new DataFrame
df_sig.head()

In [ ]:
# Generate descriptive statistics for the DataFrame and transpose the result
df_sig.describe().T

In [ ]:
# Display information about the DataFrame including the data type of each column and memory usage
df_sig.info()

In [ ]:
# Check if each column in the DataFrame is of object type, and if so, print value counts
for col in df_sig.columns:
    if pd.api.types.is_object_dtype(df_sig[col]):
        print(df_sig[col].value_counts())
        print()

In [ ]:
# Create dummy variables for all categorical columns in the DataFrame
df1 = pd.get_dummies(df_sig)

# Display information about the DataFrame with dummy variables
df1.info()

### Model Development
- Several regression algorithms were explored, including Linear Regression, Ridge Regression, Lasso Regression, ElasticNet, and Random Forest Regression.
- Each algorithm was trained on the preprocessed dataset and evaluated based on accuracy scores and mean squared errors (MSEs).

In [ ]:
# Define the feature matrix X by dropping the target column
X = df1.drop('number_of_households_engaged_majorly_in_farm_activities', axis=1)

# Define the target variable y
y = df1['number_of_households_engaged_majorly_in_farm_activities']

In [ ]:
# Split the dataset into training and testing sets with 80% for training and 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
# Initialize linear regression model
linear_regression_model = LinearRegression(
    fit_intercept=True,
    n_jobs=-1,
)

# Initialize Ridge regression model
ridge_regression_model = Ridge(
    max_iter=1000,
    random_state=1920
)

# Initialize Lasso regression model
lasso_regression_model = Lasso(
    max_iter=1000,
    random_state=1920
)

# Initialize ElasticNet regression model
elastic_net_model = ElasticNet(
    random_state=1920
)

# Initialize Random Forest regression model
random_forest_model = RandomForestRegressor(
    criterion='friedman_mse',
    verbose=True,
    random_state=1920
)

# Create a list of regression models
algos = [linear_regression_model, ridge_regression_model, lasso_regression_model, elastic_net_model, random_forest_model]

### Model Evaluation
- The performance of each model was assessed using accuracy scores and MSEs on a held-out test set.
- Visualizations, such as bar plots, were used to compare the performance of different algorithms.

In [ ]:
# Initialize empty lists to store scores, mean squared errors, and algorithm names
scores = []
mses = []
algo_names = []

# Iterate over regression algorithms
for algo in tqdm(algos):
    # Fit the algorithm on the training data
    algo.fit(X_train, y_train)
    
    # Get the name of the algorithm
    algo_name = get_var_name(algo)
    
    # Save the trained model
    dump(value=algo, filename=f'{algo_name}.joblib')
    
    # Make predictions on the testing data
    y_pred = algo.predict(X_test)
    
    # Calculate accuracy score
    acc_score = algo.score(X_test, y_test)
    
    # Calculate mean squared error
    mse = mean_squared_error(y_test, y_pred)
    
    # Append algorithm name, score, and MSE to respective lists
    algo_names.append(algo_name)
    scores.append(acc_score)
    mses.append(mse)

# Create a DataFrame to store algorithm names, scores, and MSEs
algo_scores = pd.DataFrame({
    'Algorithm': algo_names,
    'Score': scores,
    'MSE': mses
})

In [ ]:
# Initialize empty lists to store scores, mean squared errors, and algorithm names
scores = []
mses = []
algo_names = []

# Iterate over regression algorithms
for algo in tqdm(algos):
    # Get the name of the algorithm
    algo_name = get_var_name(algo)
    
    # Load the pre-trained model
    algo = load(filename=f'{algo_name}.joblib')
    
    # Make predictions on the testing data
    y_pred = algo.predict(X_test)
    
    # Calculate accuracy score
    acc_score = algo.score(X_test, y_test)
    
    # Calculate mean squared error
    mse = mean_squared_error(y_test, y_pred)
    
    # Append algorithm name, score, and MSE to respective lists
    algo_names.append(algo_name)
    scores.append(acc_score)
    mses.append(mse)

# Create a DataFrame to store algorithm names, scores, and MSEs
algo_scores = pd.DataFrame({
    'Algorithm': algo_names,
    'Score': scores,
    'MSE': mses
})

In [ ]:
# Create subplots with two rows and figsize of 15x10
fig, axs = plt.subplots(nrows=2, figsize=(15, 10))

# Plot barplot for scores
sns.barplot(
    x='Algorithm',
    y='Score',
    data=algo_scores,
    ax=axs[0],
    palette='viridis',
    legend=False,
    hue='Algorithm'
)

# Add labels to bars in the first subplot
for i in axs[0].containers:
    axs[0].bar_label(i)

# Plot barplot for MSEs
sns.barplot(
    x='Algorithm',
    y='MSE',
    data=algo_scores,
    ax=axs[1],
    palette='mako',
    legend=False,
    hue='Algorithm'
)

# Add labels to bars in the second subplot
for i in axs[1].containers:
    axs[1].bar_label(i)

# Show the plot
plt.show()

## Results
- The Random Forest Regression model exhibited the highest accuracy score and the lowest MSE among the algorithms evaluated.
- Insights gained from the analysis can inform policymakers and stakeholders about factors influencing farm household engagement and guide targeted interventions to support agricultural development.

## Conclusion
This project demonstrates the application of machine learning techniques to analyze agricultural data and predict farm household engagement based on various indicators. The developed models provide valuable insights into agricultural dynamics and can support evidence-based decision-making in agricultural policy and planning.